# Cu-Ni Phase Diagram

This example demonstrates how to build a phase diagram for the Cu-Ni system using the `SGTENode` piecewise polynomial implementation.

In [1]:
import os, sys
# Ensure local src directories are prioritized over pip-installed packages
root_dir = os.getcwd() if not os.getcwd().endswith('examples') else os.path.abspath('../../')
sys.path.insert(0, os.path.join(root_dir, 'zgraph', 'src'))
sys.path.insert(0, os.path.join(root_dir, 'thermograph', 'src'))
sys.path.insert(0, root_dir)

import jax
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from zgraph import *
from thermograph.nodes.sgte import SGTENode
from thermograph.prediction import PhaseBoundaryPredictor


## 1. Defining the Domain Physics
We extract the piecewise lattice stabilities and compile them into `zgraph` execution nodes.

In [2]:
# 1. Coordinate Signals
T, mu_Cu, mu_Ni = SignalNodes(0, 1, 2)
R = 8.314
RT = FactorNode([[R]], [T])

# 2. Build SGTENodes for each pure component phase
cu_fcc_node = SGTENode(GHSERCU, T_index=0).compile_zgraph_engine()
cu_liq_node = SGTENode(GLIQCU, T_index=0).compile_zgraph_engine()

ni_fcc_node = SGTENode(GHSERNI, T_index=0).compile_zgraph_engine()
ni_liq_node = SGTENode(GLIQNI, T_index=0).compile_zgraph_engine()

## 2. PGM Topology
We assemble the phases and the root system.

In [3]:
# 3. Construct the Subsystems (Phases)
# For ideal mixing: w_i = mu_i - G_i(T)
w_cu_fcc = FactorNode([[1.0, -1.0]], [mu_Cu, cu_fcc_node])
w_ni_fcc = FactorNode([[1.0, -1.0]], [mu_Ni, ni_fcc_node])
phase_FCC = FactorNode(jnp.eye(2), [w_cu_fcc, w_ni_fcc], beta=RT)

w_cu_liq = FactorNode([[1.0, -1.0]], [mu_Cu, cu_liq_node])
w_ni_liq = FactorNode([[1.0, -1.0]], [mu_Ni, ni_liq_node])
phase_LIQ = FactorNode(jnp.eye(2), [w_cu_liq, w_ni_liq], beta=RT)

system = FactorNode(jnp.eye(2), [phase_FCC, phase_LIQ], beta=0.0)
fcns = [phase_FCC, phase_LIQ, system]

## 3. Find the Equilibrium Manifold (Gauge Fix)
Since ZGraph natively operates in implicit state variables, our first step is to constrain the system to the equilibrium manifold using `gauge_fix`.

In [4]:
fcns_compiled = graph_to_function(fcns, compile=True)

T_vals = jnp.linspace(1200, 1800, 150)
mu_diff = jnp.linspace(-150000, 150000, 300)

T_grid, mu_grid = jnp.meshgrid(T_vals, mu_diff, indexing='ij')

inputs = jnp.stack([T_grid, mu_grid/2, -mu_grid/2], axis=-1)
inputs_flat = inputs.reshape(-1, 3)

shifted_inputs_flat = gauge_fix(fcns_compiled[2], inputs_flat, [1, 2])

## 4. Grand Potential Surface
We can visualize the raw grand potential $\Omega(T, \Delta\mu)$ for the individual phases and the total system.

In [5]:
g_vals_list_flat = [np.asarray(f(inputs_flat)).squeeze() for f in fcns_compiled]
g_vals_list = [g.reshape(150, 300) for g in g_vals_list_flat]

names = ['FCC', 'Liquid', 'System']
colorscales = ['Blues', 'Reds', 'Greens']

fig_omega = go.Figure()
T_np = np.asarray(T_grid)
mu_np = np.asarray(mu_grid)

for name, g_vals, colorscale in zip(names, g_vals_list, colorscales):
    fig_omega.add_trace(go.Surface(
        x=mu_np,
        y=T_np,
        z=g_vals,
        name=name,
        colorscale=colorscale,
        opacity=0.6,
        showscale=False
    ))

fig_omega.update_layout(
    title='Raw Grand Potential vs T and Î”Î¼',
    scene=dict(
        xaxis_title='Chemical Potential Diff (Î”Î¼)',
        yaxis_title='Temperature (K)',
        zaxis_title='Grand Potential (Î©)',
    ),
    width=900,
    height=800,
)
fig_omega.show()

## 5. Legendre Transform
With our coordinates anchored, we apply the Legendre transform to map chemical potentials to conjugate mole fractions.

In [6]:
lt_fcns = legendre_transform(fcns, [1, 2])
lt_fcns_compiled = graph_to_function(lt_fcns, compile=True)

system_lt_vals_flat = lt_fcns_compiled[2](shifted_inputs_flat)
free_energy_flat, dual_coords_flat = [np.asarray(x).squeeze() for x in system_lt_vals_flat]

free_energy = free_energy_flat.reshape(150, 300)
dual_coords = dual_coords_flat.reshape(150, 300, 3)

## 6. T-x Phase Diagram Extraction
We use the domain-specific `PhaseBoundaryPredictor` to directly extract the tie-lines.

In [7]:
predictor = PhaseBoundaryPredictor(system)

batched_x = predictor.predict_compositions(inputs, mu_index=2) # 2 is mu_Ni
x_1 = batched_x[:, 0]
x_2 = batched_x[:, 1]

x_left = np.minimum(x_1, x_2)
x_right = np.maximum(x_1, x_2)
T_vals_np = np.asarray(T_vals)

In [8]:
fig = go.Figure()

# 1. FCC Region Polygon (from x=0 to x_left)
fig.add_trace(go.Scatter(
    x=np.concatenate([np.zeros_like(x_left), x_left[::-1]]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(240, 128, 128, 0.5)', # lightcoral
    line=dict(color='rgba(255,255,255,0)'),
    name='FCC Region'
))

# 2. Liquid Region Polygon (from x_right to x=1.0)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_right, np.ones_like(x_right)]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(173, 216, 230, 0.5)', # lightblue
    line=dict(color='rgba(255,255,255,0)'),
    name='Liquid Region'
))

# 3. Two-Phase Region Polygon (between x_left and x_right)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_left, x_right[::-1]]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(211, 211, 211, 0.5)', # lightgray
    line=dict(color='rgba(255,255,255,0)'),
    name='FCC + LIQ Region'
))

fig.add_trace(go.Scatter(x=x_left, y=T_vals_np, mode='lines', name='Solidus', line=dict(color='red', width=3)))
fig.add_trace(go.Scatter(x=x_right, y=T_vals_np, mode='lines', name='Liquidus', line=dict(color='blue', width=3)))

fig.update_layout(
    title='Cu-Ni T-x Phase Diagram',
    xaxis_title='Mole Fraction Ni',
    yaxis_title='Temperature (K)',
    width=800,
    height=600
)
fig.show()